# Pipeline ANP — run agendada (Fabric)

Notebook de entrada do processamento de auditorias ANP no Fabric. É o item que o Data
Pipeline agendado (`pl_anp_pipeline_run`) dispara periodicamente.

Cada run, nessa ordem: (1) aplica lotes da Batch API da Anthropic já concluídos, (2) resolve
a fila de pendências, (3) varre `Files/entrada/` por PDFs/CSV/ZIP novos e processa.

**Antes da primeira run**, confira o runbook no `README.md` do repositório: provisionamento
do Lakehouse/Fabric SQL Database/Variable Library/Key Vault, e onde publicar `db.py`,
`pipeline.py` e `fabric_pipeline.py` (este notebook só orquestra — a lógica mora lá).

## 1. Dependências

`pyodbc`, `azure-identity`, `pydantic`, `anthropic`, `pdfplumber` e `pandas` — idealmente já
instaladas num Fabric Environment customizado anexado a este notebook (mais rápido e
reprodutível do que `%pip install` a cada run). O fallback abaixo cobre o caso de rodar sem
esse Environment.

In [ ]:
%pip install -q pyodbc azure-identity pydantic anthropic pdfplumber pandas

# Fabric AI Functions (só necessário se LLM_PROVIDER="fabric_ai") — não vem do PyPI comum,
# ver "AI Functions: Transform data at scale with AI" na doc do Fabric pro comando de wheel
# mais atual (aka.ms/fabric-aifunctions-whl). Descomente se for usar o caminho fabric_ai:
# !wget -q https://aka.ms/fabric-aifunctions-whl -O synapseml_internal-latest-py3-none-any.whl
# !wget -q https://aka.ms/fabric-synapseml-core-whl -O synapseml_core-latest-py3-none-any.whl
# %pip install -q synapseml_internal-latest-py3-none-any.whl synapseml_core-latest-py3-none-any.whl

## 2. Configuração

Lê da Variable Library do Fabric (config não-secreta) e de uma conexão Key Vault (API key da
Anthropic), e popula variáveis de ambiente — `fabric_pipeline.py`/`db.py` leem só de
`os.environ`, então isso precisa rodar ANTES de importar esses módulos.

In [ ]:
import os

# Substitua pelas chamadas reais da Variable Library / notebookutils do seu workspace, ex:
#   import notebookutils
#   os.environ["ANTHROPIC_API_KEY"] = notebookutils.credentials.getSecret("<key-vault-url>", "anthropic-api-key")

os.environ.setdefault("FABRIC_SQL_SERVER", "<seu-servidor>.database.fabric.microsoft.com")
os.environ.setdefault("FABRIC_SQL_DATABASE", "sqldb_anp_auditorias")
os.environ.setdefault("LAKEHOUSE_FILES_ROOT", "/lakehouse/default/Files")
os.environ.setdefault("LLM_PROVIDER", "claude")  # "claude" ou "fabric_ai"
os.environ.setdefault("ANTHROPIC_MODEL", "claude-haiku-4-5-20251001")
# os.environ["ANTHROPIC_API_KEY"] = ...  # via Key Vault, nunca em texto plano aqui

## 3. Importar os módulos do pipeline

Publique `db.py`, `pipeline.py` e `fabric_pipeline.py` em `Files/src/` no Lakehouse (ou anexe
como arquivos de recurso do notebook) — o `sys.path.insert` abaixo os torna importáveis.

In [ ]:
import sys

sys.path.insert(0, os.path.join(os.environ["LAKEHOUSE_FILES_ROOT"], "src"))

import fabric_pipeline

## 4. Rodar

In [ ]:
resultado = fabric_pipeline.run_pipeline()
resultado